# Stage 1 — Compare and Late-Fuse Validation Predictions

This notebook uses only the `val_predictions.csv` files created by notebooks 01–03.
No dataset CSV, manifest, split module, or model weights are loaded here.

Search of fusion weights and threshold is validation-only. Do not use `test.csv` here.

Compare/fuse one persistent output variant at a time. Set `RESULT_VARIANT='dlc'` for A-series or, for example, `RESULT_VARIANT='dlc_ccd_or_200'` for the B-series hard-negative run. W&B uses `MyDrive/Blackbox-Detection/wandb_key.txt`.

**Colab environment note:** this version does not install the full `pyproject.toml` dependency set into the live kernel. It preserves Colab's NumPy/SciPy/PyTorch stack, installs only Stage 1 extras, then installs the repository with `--no-deps` to avoid binary-package mismatch errors such as `No module named numpy.rec`.


## 1. Setup

In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

# ============================================================
# Colab + GitHub + Google Drive setup
# ============================================================
REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage1-sangchun"

IN_COLAB = False
try:
    from google.colab import drive

    IN_COLAB = True
    drive.mount("/content/drive", force_remount=False)
except ModuleNotFoundError:
    print("Not running in Google Colab; Drive mount skipped.")

if IN_COLAB:
    REPO_ROOT = Path("/content/Blackbox-Detection")

    if not (REPO_ROOT / ".git").is_dir():
        print(f"Cloning {REPO_URL} [{BRANCH}] -> {REPO_ROOT}")
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                BRANCH,
                "--single-branch",
                REPO_URL,
                str(REPO_ROOT),
            ],
            check=True,
        )
    else:
        current_branch = subprocess.run(
            ["git", "-C", str(REPO_ROOT), "branch", "--show-current"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip()

        if current_branch != BRANCH:
            subprocess.run(
                ["git", "-C", str(REPO_ROOT), "checkout", BRANCH],
                check=True,
            )

        dirty = subprocess.run(
            ["git", "-C", str(REPO_ROOT), "status", "--porcelain"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip()

        if dirty:
            print(
                "WARNING: /content/Blackbox-Detection has local changes. "
                "Automatic git pull is skipped so they are not overwritten."
            )
        else:
            print(f"Updating branch {BRANCH}...")
            subprocess.run(
                ["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", BRANCH],
                check=True,
            )
else:
    REPO_ROOT = Path.cwd().resolve()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():
        REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / "pyproject.toml").is_file():
        raise FileNotFoundError(
            "Blackbox-Detection repository not found. "
            "Run this notebook inside the repository."
        )

os.chdir(REPO_ROOT)

# ============================================================
# IMPORTANT: Colab environment policy
# ============================================================
# Do NOT run `pip install -e .` with project dependencies in a live Colab
# kernel. pyproject.toml pins NumPy/SciPy/PyTorch versions; changing those
# binary packages after the kernel has already imported them can leave a
# mixed/broken scientific stack (e.g. `No module named numpy.rec`).
#
# Instead:
#   1) keep Colab's preinstalled NumPy/SciPy/PyTorch stack,
#   2) install only the extra packages Stage 1 needs,
#   3) install this repository editable with --no-deps.

COLAB_EXTRAS = [
    "av>=15,<17",
    "timm==1.0.15",
    "fvcore==0.1.5.post20221221",
    "iopath==0.1.10",
    "yacs==0.1.8",
    "einops==0.8.1",
    "transformers==4.57.6",
    "accelerate==1.9.0",
    "huggingface-hub==0.34.4",
    "safetensors==0.6.2",
    "sentencepiece==0.2.0",
    "tokenizers>=0.22,<0.24",
    "omegaconf==2.3.0",
    "hydra-core==1.3.2",
    "wandb==0.29.0",
    "easydict==1.13",
]

if IN_COLAB:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade-strategy",
            "only-if-needed",
            *COLAB_EXTRAS,
        ],
        check=True,
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "-e",
        str(REPO_ROOT),
    ],
    check=True,
)

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

# Import the scientific stack only AFTER package setup.
try:
    import numpy as np
    import pandas as pd
    import scipy
    from scipy.ndimage import maximum_filter
    import torch
    import yaml
except Exception as exc:
    raise RuntimeError(
        "The Colab scientific Python stack is inconsistent. "
        "This usually happens if NumPy/SciPy were changed earlier in the same "
        "runtime. Use Runtime -> Restart session once, then run this notebook "
        "again from the top. Do not run the old notebook's `pip install -e .` "
        "cell before restarting."
    ) from exc

from blackbox_detection.utils import (
    finish_wandb,
    init_wandb,
    load_checkpoint,
    seed_everything,
    setup_logger,
)

# ============================================================
# Persistent Google Drive paths
# ============================================================
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATASET_ROOT = DRIVE_PROJECT_ROOT / "DATASET"
DLC_ROOT = DATASET_ROOT / "DLC-2021"
CCD_ROOT = DATASET_ROOT / "CCD"

DLC_SPLIT_CSV = DLC_ROOT / "dlc_split.csv"
CCD_SPLIT_CSV = CCD_ROOT / "ccd_split.csv"

OUTPUT_ROOT = DRIVE_PROJECT_ROOT / "outputs" / "stage1"
WANDB_KEY_PATH = DRIVE_PROJECT_ROOT / "wandb_key.txt"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    required_paths = {
        "DATASET_ROOT": DATASET_ROOT,
        "DLC_ROOT": DLC_ROOT,
        "CCD_ROOT": CCD_ROOT,
        "DLC_SPLIT_CSV": DLC_SPLIT_CSV,
        "CCD_SPLIT_CSV": CCD_SPLIT_CSV,
        "WANDB_KEY_PATH": WANDB_KEY_PATH,
    }
    missing = [
        f"{name}: {path}"
        for name, path in required_paths.items()
        if not path.exists()
    ]
    if missing:
        raise FileNotFoundError(
            "Required Drive paths are missing:\n  " + "\n  ".join(missing)
        )

# ============================================================
# Weights & Biases login
# ============================================================
WANDB_ENABLED = True
WANDB_PROJECT = "blackbox-stage1"
WANDB_ENTITY = os.getenv("WANDB_ENTITY") or None
WANDB_MODE = os.getenv("WANDB_MODE") or None

if WANDB_ENABLED:
    import wandb

    if IN_COLAB:
        wandb_key = WANDB_KEY_PATH.read_text(encoding="utf-8").strip()
        if not wandb_key:
            raise ValueError(f"W&B key file is empty: {WANDB_KEY_PATH}")
        wandb.login(key=wandb_key, relogin=False)
        del wandb_key
    else:
        wandb.login()

GIT_COMMIT = subprocess.run(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "--short", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

print("REPO_ROOT          :", REPO_ROOT)
print("branch / commit    :", BRANCH, "/", GIT_COMMIT)
print("DATASET_ROOT       :", DATASET_ROOT)
print("DLC split          :", DLC_SPLIT_CSV)
print("CCD split          :", CCD_SPLIT_CSV)
print("persistent outputs :", OUTPUT_ROOT)
print("numpy              :", np.__version__)
print("scipy              :", scipy.__version__)
print("torch              :", torch.__version__, "| cuda:", torch.cuda.is_available())
print("W&B                :", "enabled" if WANDB_ENABLED else "disabled")


## 2. Paths

In [ ]:
# Choose the training-data variant whose predictions should be compared.
#
# A-series:
RESULT_VARIANT = "dlc"
#
# B-series examples:
# RESULT_VARIANT = "dlc_ccd_or_200"
# RESULT_VARIANT = "dlc_ccd_or_all"

RESULT_ROOT = OUTPUT_ROOT / RESULT_VARIANT
FUSION_DIR = RESULT_ROOT / "fusion"
FUSION_DIR.mkdir(parents=True, exist_ok=True)

print("result variant :", RESULT_VARIANT)
print("result root    :", RESULT_ROOT)
print("fusion output  :", FUSION_DIR)

finish_wandb()
wandb_run = init_wandb(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=f"fusion_compare__{RESULT_VARIANT}__{GIT_COMMIT}",
    group=f"fusion__{RESULT_VARIANT}",
    tags=["stage1", "fusion", "comparison", RESULT_VARIANT],
    config={
        "git_commit": GIT_COMMIT,
        "result_variant": RESULT_VARIANT,
        "result_root": str(RESULT_ROOT),
    },
    directory=FUSION_DIR / "wandb",
    mode=WANDB_MODE,
)
print("W&B run:", wandb_run.name)
print("W&B url :", wandb_run.url)


## 3. Available validation predictions

In [ ]:
CANDIDATES = {
    "videomaev2_b": "V1 VideoMAEv2-B",
    "vjepa2_1_b": "V2 V-JEPA 2.1-B",
    "bayar_resnet18": "F1 Bayar + ResNet18",
    "cdc": "F2 CDC",
    "chromaticity": "F3 CMA-inspired chromaticity",
    "frequency": "F4 Frequency/Moiré",
    "lcdf": "F5 LC&DF-inspired",
}

PREDICTION_PATHS = {
    key: RESULT_ROOT / key / "val_predictions.csv"
    for key in CANDIDATES
    if (RESULT_ROOT / key / "val_predictions.csv").is_file()
}

if not PREDICTION_PATHS:
    raise FileNotFoundError(
        "No val_predictions.csv found. Run notebooks 01/02/03 first."
    )

print("available:")
for key, path in PREDICTION_PATHS.items():
    print(f"  {key:18s} -> {path}")

## 4. Load / compare

In [ ]:
wide = load_prediction_tables(PREDICTION_PATHS)
print("validation videos:", len(wide))
print("class balance:", wide["label"].value_counts().to_dict())

comparison = compare_models(wide)
comparison.insert(1, "description", comparison["model"].map(CANDIDATES))
display(comparison)

if WANDB_ENABLED and wandb_run is not None:
    wandb_run.log({"single_model_comparison": wandb.Table(dataframe=comparison)})


In [ ]:
if len(PREDICTION_PATHS) >= 2:
    correlation = prediction_correlation(wide)
    display(correlation)
    if WANDB_ENABLED and wandb_run is not None:
        wandb_run.log({"prediction_correlation": wandb.Table(dataframe=correlation.reset_index())})


## 5. Late fusion

In [ ]:
available = list(PREDICTION_PATHS)
rows = []
results = {}

# Pair searches are cheap and directly show complementarity.
for pair in combinations(available, 2):
    result = search_late_fusion(wide, pair, weight_step=0.05)
    results[pair] = result
    rows.append(result.as_row())

ensemble_table = (
    pd.DataFrame(rows)
    .sort_values("macro_f1", ascending=False, kind="mergesort")
    .reset_index(drop=True)
    if rows
    else pd.DataFrame()
)
display(ensemble_table)

if WANDB_ENABLED and wandb_run is not None and not ensemble_table.empty:
    wandb_run.log({"pair_fusion_search": wandb.Table(dataframe=ensemble_table)})


In [ ]:
# Optional three-model search: two video backbones + best available forensic model.
video_models = [m for m in ("videomaev2_b", "vjepa2_1_b") if m in available]
forensic_models = [
    m for m in ("lcdf", "chromaticity", "frequency", "bayar_resnet18", "cdc")
    if m in available
]

three_model_result = None
if len(video_models) == 2 and forensic_models:
    forensic_rank = (
        comparison[comparison["model"].isin(forensic_models)]
        .sort_values("macro_f1", ascending=False)
    )
    best_forensic = forensic_rank.iloc[0]["model"]
    group = (*video_models, best_forensic)
    three_model_result = search_late_fusion(wide, group, weight_step=0.05)
    print("three-model candidate:", group)
    display(pd.DataFrame([three_model_result.as_row()]))

## 6. Save best validation fusion

In [ ]:
all_results = list(results.values())
if three_model_result is not None:
    all_results.append(three_model_result)

if not all_results:
    print("Need at least two validation prediction files for fusion.")
else:
    best = max(all_results, key=lambda result: result.macro_f1)
    best_predictions = fused_predictions(wide, best)

    save_predictions(best_predictions.drop(columns=["threshold"]), FUSION_DIR / "val_predictions.csv")
    (FUSION_DIR / "fusion_summary.json").write_text(
        json.dumps(
            {
                "models": list(best.models),
                "weights": list(best.weights),
                "threshold": float(best.threshold),
                "macro_f1": float(best.macro_f1),
                "gain_over_best_single": float(best.gain_over_best_single),
                "per_class_f1": best.per_class_f1,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    print("best fusion:", best.as_row())
    print("saved to:", FUSION_DIR)

    if WANDB_ENABLED and wandb_run is not None:
        wandb_run.summary["best_models"] = list(best.models)
        wandb_run.summary["best_weights"] = list(best.weights)
        wandb_run.summary["best_threshold"] = float(best.threshold)
        wandb_run.summary["best_macro_f1"] = float(best.macro_f1)
        wandb_run.summary["gain_over_best_single"] = float(best.gain_over_best_single)
        for class_name, score in best.per_class_f1.items():
            wandb_run.summary[f"best_f1_{class_name.lower()}"] = float(score)

        try:
            wandb_run.save(
                str(FUSION_DIR / "fusion_summary.json"),
                base_path=str(FUSION_DIR),
            )
            wandb_run.save(
                str(FUSION_DIR / "val_predictions.csv"),
                base_path=str(FUSION_DIR),
            )
        except Exception as exc:
            print("W&B file upload warning:", exc)

finish_wandb()
print("W&B fusion run finished.")


## Important

`test.csv` is not used here. Fusion weights and the decision threshold are selected on validation only.
After the architecture/weights are frozen, apply those fixed choices once to the internal test split.